# 2026 EY AI & Data Challenge — Improved Baseline Model

This notebook builds an improved baseline for predicting:
- Total Alkalinity
- Electrical Conductance
- Dissolved Reactive Phosphorus

It uses only the provided challenge files and generates a valid submission CSV.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

In [2]:
TRAIN_PATH = 'water_quality_training_dataset.csv'
SUBMISSION_TEMPLATE_PATH = 'submission_template.csv'
OUTPUT_PATH = 'submission_improved.csv'

TARGET_COLUMNS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

train_df = pd.read_csv(TRAIN_PATH)
submission_df = pd.read_csv(SUBMISSION_TEMPLATE_PATH)

for df in (train_df, submission_df):
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], format='%d-%m-%Y', errors='coerce')

print('Training rows:', len(train_df))
print('Submission rows:', len(submission_df))
print('Missing dates in training:', train_df['Sample Date'].isna().sum())
train_df.head()

Training rows: 9319
Submission rows: 200
Missing dates in training: 0


,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,2011-01-02,128.912,555.0,10.0
1,-26.861111,28.884722,2011-01-03,74.720,162.9,163.0
2,-26.450000,28.085833,2011-01-03,89.254,573.0,80.0
3,-27.671111,27.236944,2011-01-03,82.000,203.6,101.0
4,-27.356667,27.286389,2011-01-03,56.100,145.1,151.0


In [3]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return r * c

CITY_COORDS = {
    'cape_town': (-33.9249, 18.4241),
    'johannesburg': (-26.2041, 28.0473),
    'pretoria': (-25.7479, 28.2293),
    'durban': (-29.8587, 31.0218),
}

def build_features(df):
    out = pd.DataFrame(index=df.index)

    out['lat'] = df['Latitude'].astype(float)
    out['lon'] = df['Longitude'].astype(float)

    out['year'] = df['Sample Date'].dt.year.fillna(0).astype(int)
    out['month'] = df['Sample Date'].dt.month.fillna(0).astype(int)
    out['day'] = df['Sample Date'].dt.day.fillna(0).astype(int)
    out['dayofyear'] = df['Sample Date'].dt.dayofyear.fillna(0).astype(int)
    out['quarter'] = df['Sample Date'].dt.quarter.fillna(0).astype(int)

    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
    out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 365.25)

    out['lat_lon'] = out['lat'] * out['lon']
    out['lat_sq'] = out['lat'] ** 2
    out['lon_sq'] = out['lon'] ** 2

    for city, (city_lat, city_lon) in CITY_COORDS.items():
        out[f'dist_{city}_km'] = haversine_km(out['lat'].values, out['lon'].values, city_lat, city_lon)

    dist_cols = [c for c in out.columns if c.startswith('dist_')]
    out['dist_min_major_city_km'] = out[dist_cols].min(axis=1)

    out['station_key'] = (
        df['Latitude'].round(5).astype(str) + '_' + df['Longitude'].round(5).astype(str)
    )

    return out

X_train_full = build_features(train_df)
X_submit_full = build_features(submission_df)

feature_cols = [c for c in X_train_full.columns if c != 'station_key']
X_train = X_train_full[feature_cols].copy()
X_submit = X_submit_full[feature_cols].copy()

groups = X_train_full['station_key'].values

X_train.head()

,lat,lon,year,month,day,dayofyear,quarter,month_sin,month_cos,doy_sin,doy_cos,lat_lon,lat_sq,lon_sq,dist_cape_town_km,dist_johannesburg_km,dist_pretoria_km,dist_durban_km,dist_min_major_city_km
0,-28.760833,17.730278,2011,1,2,2,1,0.5,0.866025,0.034398,0.999408,-509.937565,827.185515,314.362758,577.980774,1056.272910,1090.071153,1293.789916,577.980774
1,-26.861111,28.884722,2011,1,3,3,1,0.5,0.866025,0.051584,0.998669,-775.875724,721.519284,834.327165,1272.903489,110.804434,139.966058,393.456427,110.804434
2,-26.450000,28.085833,2011,1,3,3,1,0.5,0.866025,0.051584,0.998669,-742.870283,699.602500,788.814015,1245.082687,27.611194,79.373506,475.883015,27.611194
3,-27.671111,27.236944,2011,1,3,3,1,0.5,0.866025,0.051584,0.998669,-753.676501,765.690384,741.851118,1091.003104,181.829304,235.471715,441.850365,181.829304
4,-27.356667,27.286389,2011,1,3,3,1,0.5,0.866025,0.051584,0.998669,-746.464658,748.387229,744.547025,1118.160374,148.761385,201.979384,458.609223,148.761385


In [4]:
def build_model(random_state=42):
    rf = RandomForestRegressor(
        n_estimators=450,
        max_depth=20,
        min_samples_leaf=2,
        random_state=random_state,
        n_jobs=-1
    )

    et = ExtraTreesRegressor(
        n_estimators=650,
        max_depth=None,
        min_samples_leaf=1,
        random_state=random_state,
        n_jobs=-1
    )

    hgb = HistGradientBoostingRegressor(
        learning_rate=0.04,
        max_depth=8,
        max_iter=550,
        l2_regularization=0.05,
        random_state=random_state
    )

    return VotingRegressor(
        estimators=[('rf', rf), ('et', et), ('hgb', hgb)],
        weights=[1.0, 1.2, 1.4]
    )

def cv_r2_scores(X, y, groups, n_splits=5, transform='none'):
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(y), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), start=1):
        x_tr, x_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y.iloc[tr_idx].values

        if transform == 'log1p':
            y_fit = np.log1p(np.clip(y_tr, 0, None))
        else:
            y_fit = y_tr

        model = build_model(random_state=42 + fold)
        model.fit(x_tr, y_fit)

        pred = model.predict(x_va)
        if transform == 'log1p':
            pred = np.expm1(pred)

        oof[va_idx] = pred

    score = r2_score(y, oof)
    return score

In [5]:
TRANSFORM_OPTIONS = {
    'Total Alkalinity': ['none', 'log1p'],
    'Electrical Conductance': ['none', 'log1p'],
    'Dissolved Reactive Phosphorus': ['none', 'log1p']
}

best_transform = {}
cv_scores = {}

for target in TARGET_COLUMNS:
    y = train_df[target].astype(float)

    best_score = -1e9
    best_t = None

    for t_name in TRANSFORM_OPTIONS[target]:
        score = cv_r2_scores(X_train, y, groups, n_splits=5, transform=t_name)
        print(f'{target} | transform={t_name} | GroupCV R2={score:.4f}')

        if score > best_score:
            best_score = score
            best_t = t_name

    best_transform[target] = best_t
    cv_scores[target] = best_score

print('\nBest transforms:', best_transform)
print('Best GroupCV scores:', cv_scores)
print('Mean GroupCV R2:', float(np.mean(list(cv_scores.values()))))

Total Alkalinity | transform=none | GroupCV R2=0.1789
Total Alkalinity | transform=log1p | GroupCV R2=0.1456
Electrical Conductance | transform=none | GroupCV R2=0.2496
Electrical Conductance | transform=log1p | GroupCV R2=0.2970
Dissolved Reactive Phosphorus | transform=none | GroupCV R2=-0.0198
Dissolved Reactive Phosphorus | transform=log1p | GroupCV R2=-0.0114

Best transforms: {'Total Alkalinity': 'none', 'Electrical Conductance': 'log1p', 'Dissolved Reactive Phosphorus': 'log1p'}
Best GroupCV scores: {'Total Alkalinity': 0.1789048313509798, 'Electrical Conductance': 0.2969598251489787, 'Dissolved Reactive Phosphorus': -0.011394113973400621}
Mean GroupCV R2: 0.1548235141755193


In [6]:
final_predictions = pd.DataFrame(index=submission_df.index)
trained_models = {}

for target in TARGET_COLUMNS:
    y = train_df[target].astype(float).values
    transform = best_transform[target]

    model = build_model(random_state=2026)

    if transform == 'log1p':
        y_fit = np.log1p(np.clip(y, 0, None))
    else:
        y_fit = y

    model.fit(X_train, y_fit)
    pred = model.predict(X_submit)

    if transform == 'log1p':
        pred = np.expm1(pred)

    pred = np.clip(pred, 0, None)

    final_predictions[target] = pred
    trained_models[target] = model

final_predictions.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,136.399363,278.433121,23.198256
1,262.346293,549.104220,27.381864
2,199.670155,306.355666,41.914923
3,205.946561,411.724146,15.035931
4,112.225076,281.509774,21.553641


In [ ]:
submission_out = submission_df.copy()
for col in TARGET_COLUMNS:
    submission_out[col] = final_predictions[col].astype(float)

submission_out.to_csv(OUTPUT_PATH, index=False)
print(f'Submission file created: {OUTPUT_PATH}')
submission_out.head()

## Next improvement ideas

1. Join Landsat and TerraClimate engineered features (from the extraction notebooks) to both training and submission rows by location/date.
2. Train target-specific models with parameter search.
3. Add model ensembling across different random seeds and algorithms.
4. Validate with station-grouped folds to protect against location leakage.